# Notebook 01 — Data Normalization

**Purpose:** Convert all five instruments to a single comparable unit: annualized expected return in IDR, forward-looking, on a consistent basis.

**Inputs:** 
- `data/raw/sbn_yields.csv`
- `data/raw/rdpu_nav.csv`
- `data/raw/usd_idr_rate.csv`
- `data/raw/aave_apy.csv`
- `data/raw/pendle_pt_apy.csv`
- `data/raw/pendle_yt_apy.csv`

**Outputs:** 
- `data/processed/expected_returns.csv`
- `data/processed/expected_returns_summary.csv`

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime

# Add src to path
sys.path.append(os.path.abspath('../src'))

import normalization as norm
import utils

np.random.seed(42)
print(f"Notebook initialized at {datetime.now()}")

## 1. Load and Clean Data

We need to normalize different date formats and clean numerical strings.

In [ ]:
def clean_currency_str(s):
    if isinstance(s, str):
        return float(s.replace(',', ''))
    return s

# 1. USD/IDR Rate
fx_df = pd.read_csv('../data/raw/usd_idr_rate.csv')
fx_df['date'] = pd.to_datetime(fx_df['Date']).dt.strftime('%Y-%m-%d')
fx_df['usd_idr_rate'] = fx_df['Price'].apply(clean_currency_str)
fx_df = fx_df[['date', 'usd_idr_rate']].sort_values('date')

# 2. SBN Yields
sbn_df = pd.read_csv('../data/raw/sbn_yields.csv')
sbn_df['date'] = pd.to_datetime(sbn_df['Date']).dt.strftime('%Y-%m-%d')
sbn_df['yield_annual'] = sbn_df['Price'] / 100.0
sbn_df['instrument'] = 'sbn'
sbn_df['currency'] = 'IDR'
sbn_df = sbn_df[['date', 'instrument', 'yield_annual', 'currency']].sort_values('date')

# 3. RDPU NAV (Transcribed)
rdpu_df = pd.read_csv('../data/raw/rdpu_nav.csv')
rdpu_df['date'] = pd.to_datetime(rdpu_df['date']).dt.strftime('%Y-%m-%d')
rdpu_df = rdpu_df.sort_values('date')

# 4. Aave & Pendle (Already clean from Notebook 00)
aave_df = pd.read_csv('../data/raw/aave_apy.csv', comment='#')
pt_df = pd.read_csv('../data/raw/pendle_pt_apy.csv', comment='#')
yt_df = pd.read_csv('../data/raw/pendle_yt_apy.csv', comment='#')

print("Data loaded and initial cleaning complete.")

## 2. Annualization and Conversion

### MMF (RDPU)
Calculate rolling 30-day annualized yield from NAV.

In [ ]:
# Since rdpu is monthly/sporadic, we'll interpolate to daily first
rdpu_df = rdpu_df.set_index(pd.to_datetime(rdpu_df['date']))
rdpu_daily = rdpu_df.resample('D').interpolate(method='linear')
rdpu_daily['yield_annual'] = norm.annualize_nav_series(rdpu_daily['nav'], window=30)
rdpu_daily['instrument'] = 'mmf'
rdpu_daily['currency'] = 'IDR'
rdpu_final = rdpu_daily.reset_index().rename(columns={'index': 'date'})
rdpu_final['date'] = rdpu_final['date'].dt.strftime('%Y-%m-%d')
rdpu_final = rdpu_final[['date', 'instrument', 'yield_annual', 'currency']].dropna()

### On-Chain USD to IDR
Combine with FX rate return.

In [ ]:
def process_on_chain(df, label):
    # Join with FX rate
    merged = df.merge(fx_df, on='date', how='inner').sort_values('date')
    # fx_return over 30 days proxy for annualized adjustment
    merged['fx_return_annual'] = (merged['usd_idr_rate'] / merged['usd_idr_rate'].shift(30)) ** (365/30) - 1
    merged['yield_idr'] = (1 + merged['yield_annual']) * (1 + merged['fx_return_annual'].fillna(0)) - 1
    merged['instrument'] = label
    return merged[['date', 'instrument', 'yield_annual', 'yield_idr']]

aave_proc = process_on_chain(aave_df, 'aave')
pt_proc = process_on_chain(pt_df, 'pendle_pt')
yt_proc = process_on_chain(yt_df, 'pendle_yt')

print("On-chain conversion complete.")

## 3. Mean Reversion and Master Consolidation

**Assumption:** We apply a 0.4 spot / 0.6 trailing mean blend for variable rate instruments (`mmf`, `aave`, `pendle_yt`).

In [ ]:
# Prepare Unified DataFrame
off_chain = pd.concat([rdpu_final, sbn_df])
off_chain['yield_idr'] = off_chain['yield_annual']

master_df = pd.concat([off_chain, aave_proc, pt_proc, yt_proc])
master_df['yield_adjusted'] = master_df['yield_idr']

# Apply mean reversion for variable rates
variable_instruments = ['mmf', 'aave', 'pendle_yt']
for inst in variable_instruments:
    mask = master_df['instrument'] == inst
    inst_series = master_df[mask].sort_values('date')
    hist_mean = inst_series['yield_idr'].expanding().mean()
    master_df.loc[mask, 'yield_adjusted'] = norm.mean_reversion_blend(inst_series['yield_idr'], hist_mean)

master_df.to_csv('../data/processed/expected_returns.csv', index=False)

summary = master_df.groupby('instrument')['yield_idr'].agg(['mean', 'std', 'min', 'max'])
summary.to_csv('../data/processed/expected_returns_summary.csv')

print("Master expected returns saved.")
summary

**Sanity check:** Yields should roughly follow mmf < sbn < aave < pendle_pt < pendle_yt.